# Preprocessing — Teen Mental Health Dataset

This notebook transforms the raw dataset into a clean, model-ready format.  
All steps are applied **after** splitting the data to prevent data leakage.

### Steps
1. Load raw data  
2. Encode categorical features  
3. Train / test split (stratified)  
4. Scale numerical features  
5. Handle class imbalance with SMOTE (training set only)  
6. Save processed datasets  
7. Summary  


---
## 1. Imports and Configuration

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Class imbalance
from imblearn.over_sampling import SMOTE

# Display
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# -- Project root resolution --------------------------------------------------
# Walks up from the current working directory until it finds requirements.txt.
# This makes the notebook runnable regardless of where Jupyter was launched from
# (e.g. project root or notebooks/ subfolder).
def find_project_root(marker: str = 'requirements.txt') -> Path:
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root. "
        f"Ensure '{marker}' exists at the root of the project."
    )

PROJECT_ROOT  = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print(f'Project root  : {PROJECT_ROOT}')
print(f'Processed dir : {PROCESSED_DIR}')
print('Imports complete.')

Project root  : /Users/usuario/Desktop/logistic regression
Processed dir : /Users/usuario/Desktop/logistic regression/data/processed
Imports complete.


---
## 2. Load Raw Data

In [2]:
DATA_PATH = PROJECT_ROOT / 'data' / 'Teen_Mental_Health_Dataset.csv'

df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
display(df.head())

Shape: 1,200 rows x 13 columns
Missing values: 0


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9000,Instagram,7.4000,2.9000,3.0100,1.5000,low,2,2,1,0
1,19,female,1.9000,TikTok,8.0000,2.9000,3.2200,0.8000,high,8,1,10,0
2,17,female,1.3000,Instagram,7.6000,0.5000,3.9200,0.0000,high,2,4,2,0
3,15,male,7.4000,TikTok,6.9000,1.6000,3.4800,0.8000,medium,1,7,9,0
4,15,female,4.7000,Both,4.9000,3.0000,2.3700,1.4000,medium,3,5,2,0


---
## 3. Define Feature Groups

Each group will receive a different transformation:

| Group | Features | Transformation |
|---|---|---|
| Numerical | continuous variables | StandardScaler |
| Binary categorical | `gender` | LabelEncoder |
| Nominal categorical | `platform_usage` | OneHotEncoder |
| Ordinal categorical | `social_interaction_level` | OrdinalEncoder |

In [3]:
TARGET = 'depression_label'

# Numerical features — will be scaled with StandardScaler
NUMERICAL_FEATURES = [
    'age',
    'daily_social_media_hours',
    'sleep_hours',
    'screen_time_before_sleep',
    'academic_performance',
    'physical_activity',
    'stress_level',
    'anxiety_level',
    'addiction_level',
]

# Binary categorical — male=0, female=1
BINARY_FEATURES = ['gender']

# Nominal categorical — no intrinsic order, will be one-hot encoded
NOMINAL_FEATURES = ['platform_usage']

# Ordinal categorical — has a meaningful order
ORDINAL_FEATURES = ['social_interaction_level']
ORDINAL_CATEGORIES = [['low', 'medium', 'high']]  # ascending order

ALL_FEATURES = NUMERICAL_FEATURES + BINARY_FEATURES + NOMINAL_FEATURES + ORDINAL_FEATURES

print('Feature groups defined:')
print(f'  Numerical  ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}')
print(f'  Binary     ({len(BINARY_FEATURES)}): {BINARY_FEATURES}')
print(f'  Nominal    ({len(NOMINAL_FEATURES)}): {NOMINAL_FEATURES}')
print(f'  Ordinal    ({len(ORDINAL_FEATURES)}): {ORDINAL_FEATURES}')

Feature groups defined:
  Numerical  (9): ['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep', 'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']
  Binary     (1): ['gender']
  Nominal    (1): ['platform_usage']
  Ordinal    (1): ['social_interaction_level']


---
## 4. Train / Test Split

The split is performed **before** any transformation to prevent data leakage.  
Stratification ensures both sets maintain the original 97.4% / 2.6% class ratio.

In [4]:
X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # preserve class ratio in both sets
    random_state=RANDOM_STATE,
)

print('Split summary:')
print(f'  Training set : {X_train.shape[0]:>4} rows  |  class 0: {(y_train==0).sum()}  class 1: {(y_train==1).sum()}')
print(f'  Test set     : {X_test.shape[0]:>4} rows  |  class 0: {(y_test==0).sum()}  class 1: {(y_test==1).sum()}')

Split summary:
  Training set :  960 rows  |  class 0: 935  class 1: 25
  Test set     :  240 rows  |  class 0: 234  class 1: 6


---
## 5. Build the Preprocessing Pipeline

A `ColumnTransformer` applies different transformations to each feature group simultaneously.  
The pipeline is **fit only on the training set** and then used to transform both sets.

In [5]:
# -- Binary encoding: female=1, male=0 ----------------------------------------
binary_encoder = OrdinalEncoder(categories=[['male', 'female']])

# -- Ordinal encoding: low=0, medium=1, high=2 ---------------------------------
ordinal_encoder = OrdinalEncoder(categories=ORDINAL_CATEGORIES)

# -- One-Hot encoding for nominal categories -----------------------------------
nominal_encoder = OneHotEncoder(drop='first', sparse_output=False)

# -- ColumnTransformer: applies each transformer to its assigned columns -------
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(),    NUMERICAL_FEATURES),
        ('binary',    binary_encoder,      BINARY_FEATURES),
        ('nominal',   nominal_encoder,     NOMINAL_FEATURES),
        ('ordinal',   ordinal_encoder,     ORDINAL_FEATURES),
    ],
    remainder='drop',
)

print('Preprocessing pipeline defined.')
print('Transformers:')
print('  StandardScaler    ->  numerical features')
print('  OrdinalEncoder    ->  gender          (male=0, female=1)')
print('  OneHotEncoder     ->  platform_usage  (drop first to avoid dummy trap)')
print('  OrdinalEncoder    ->  social_interaction_level  (low=0, medium=1, high=2)')

Preprocessing pipeline defined.
Transformers:
  StandardScaler    ->  numerical features
  OrdinalEncoder    ->  gender          (male=0, female=1)
  OneHotEncoder     ->  platform_usage  (drop first to avoid dummy trap)
  OrdinalEncoder    ->  social_interaction_level  (low=0, medium=1, high=2)


---
## 6. Fit and Transform

In [6]:
# Fit on training data only — then transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

# Recover column names after transformation
nominal_cols = (
    preprocessor
    .named_transformers_['nominal']
    .get_feature_names_out(NOMINAL_FEATURES)
    .tolist()
)

FEATURE_NAMES = NUMERICAL_FEATURES + BINARY_FEATURES + nominal_cols + ORDINAL_FEATURES

# Convert back to DataFrames for readability
X_train_df = pd.DataFrame(X_train_processed, columns=FEATURE_NAMES)
X_test_df  = pd.DataFrame(X_test_processed,  columns=FEATURE_NAMES)

print(f'Training set shape after preprocessing : {X_train_df.shape}')
print(f'Test set shape after preprocessing     : {X_test_df.shape}')
print(f'\nFinal feature names ({len(FEATURE_NAMES)}):')
for name in FEATURE_NAMES:
    print(f'  - {name}')

print('\nSample of preprocessed training data:')
display(X_train_df.head())

Training set shape after preprocessing : (960, 13)
Test set shape after preprocessing     : (240, 13)

Final feature names (13):
  - age
  - daily_social_media_hours
  - sleep_hours
  - screen_time_before_sleep
  - academic_performance
  - physical_activity
  - stress_level
  - anxiety_level
  - addiction_level
  - gender
  - platform_usage_Instagram
  - platform_usage_TikTok
  - social_interaction_level

Sample of preprocessed training data:


,age,daily_social_media_hours,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,stress_level,anxiety_level,addiction_level,gender,platform_usage_Instagram,platform_usage_TikTok,social_interaction_level
0,-1.4405,-1.1569,-0.4428,0.3499,1.1051,1.5131,-0.4982,-0.5770,1.5787,0.0000,0.0000,0.0000,0.0000
1,-1.4405,1.6251,-0.3062,1.3287,-0.2530,-1.2331,1.2358,1.5145,-0.9175,0.0000,0.0000,1.0000,0.0000
2,0.0433,0.3073,1.1286,0.4897,-0.5664,1.1698,1.2358,1.5145,1.5787,1.0000,0.0000,1.0000,1.0000
3,1.5270,-0.6200,-0.9894,0.4897,-1.6632,-0.2033,-1.5386,1.5145,1.2221,1.0000,0.0000,1.0000,2.0000
4,0.5379,0.2097,-1.0577,0.4897,-1.7155,-0.0316,1.2358,-0.5770,1.2221,1.0000,0.0000,0.0000,0.0000


---
## 7. Handle Class Imbalance — SMOTE

The dataset has a severe imbalance: **97.4% class 0 vs 2.6% class 1**.  
A model trained on this raw distribution would learn to always predict class 0 and still achieve ~97% accuracy.

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class by interpolating between existing minority samples, rather than simply duplicating them.

> SMOTE is applied **only to the training set**. Applying it to the test set would produce an unrealistic evaluation, since real-world data will always be imbalanced.

In [7]:
print('Before SMOTE:')
print(f'  Class 0: {(y_train == 0).sum()}  |  Class 1: {(y_train == 1).sum()}')

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_df, y_train)

X_train_resampled = pd.DataFrame(X_train_resampled, columns=FEATURE_NAMES)
y_train_resampled = pd.Series(y_train_resampled, name=TARGET)

print('\nAfter SMOTE:')
print(f'  Class 0: {(y_train_resampled == 0).sum()}  |  Class 1: {(y_train_resampled == 1).sum()}')
print(f'  Total training samples: {len(X_train_resampled)}')

Before SMOTE:
  Class 0: 935  |  Class 1: 25

After SMOTE:
  Class 0: 935  |  Class 1: 935
  Total training samples: 1870


---
## 8. Save Processed Datasets

Four files are saved to `data/processed/` — they are gitignored (reproducible from this notebook):  

| File | Description |
|---|---|
| `X_train.csv` | Training features (scaled, encoded, balanced via SMOTE) |
| `X_test.csv` | Test features (scaled and encoded, original distribution) |
| `y_train.csv` | Training labels (balanced via SMOTE) |
| `y_test.csv` | Test labels (original distribution) |

In [8]:
X_train_resampled.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test_df.to_csv(        PROCESSED_DIR / 'X_test.csv',  index=False)
y_train_resampled.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(           PROCESSED_DIR / 'y_test.csv',  index=False)

print('Files saved to data/processed/:')
for fname in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']:
    fpath = PROCESSED_DIR / fname
    size  = fpath.stat().st_size / 1024
    print(f'  {fname:<15}  {size:6.1f} KB')

Files saved to data/processed/:
  X_train.csv       374.1 KB
  X_test.csv         45.3 KB
  y_train.csv         3.7 KB
  y_test.csv          0.5 KB


---
## 9. Summary

In [9]:
print('=' * 60)
print('          PREPROCESSING SUMMARY')
print('=' * 60)
print(f'  Raw dataset shape          : {df.shape}')
print(f'  Final feature count        : {len(FEATURE_NAMES)}')
print()
print('  Encoding applied:')
print('    gender                   : OrdinalEncoder  (male=0, female=1)')
print('    platform_usage           : OneHotEncoder   (drop first)')
print('    social_interaction_level : OrdinalEncoder  (low=0, medium=1, high=2)')
print()
print('  Scaling applied:')
print('    All numerical features   : StandardScaler')
print()
print('  Split (stratified, test_size=0.2):')
print(f'    Training : {X_train.shape[0]} rows')
print(f'    Test     : {X_test.shape[0]} rows')
print()
print('  Class imbalance (SMOTE on training set only):')
print(f'    Before : class 0={( y_train==0).sum()}  class 1={(y_train==1).sum()}')
print(f'    After  : class 0={(y_train_resampled==0).sum()}  class 1={(y_train_resampled==1).sum()}')
print('=' * 60)
print()
print('Next step: notebooks/03_model.ipynb')

          PREPROCESSING SUMMARY
  Raw dataset shape          : (1200, 13)
  Final feature count        : 13

  Encoding applied:
    gender                   : OrdinalEncoder  (male=0, female=1)
    platform_usage           : OneHotEncoder   (drop first)
    social_interaction_level : OrdinalEncoder  (low=0, medium=1, high=2)

  Scaling applied:
    All numerical features   : StandardScaler

  Split (stratified, test_size=0.2):
    Training : 960 rows
    Test     : 240 rows

  Class imbalance (SMOTE on training set only):
    Before : class 0=935  class 1=25
    After  : class 0=935  class 1=935

Next step: notebooks/03_model.ipynb
